# 01 - EDA: text overview

Text-only EDA on `Train.csv`/`Test.csv` (no images required). Image-level EDA (sizes, aspect ratios, quality) belongs in a later notebook once `data/raw/images/` is populated.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data import load_raw_data

train = load_raw_data('train')
test = load_raw_data('test')
train.shape, test.shape

((4098, 2), (1374, 1))

In [2]:
train['Target'].str.len().describe()

count    4098.000000
mean       62.278184
std        12.705858
min        37.000000
25%        52.000000
50%        62.000000
75%        71.000000
max       120.000000
Name: Target, dtype: float64

In [3]:
# Character set coverage - flag anything outside plain ASCII letters/punctuation
chars = sorted(set(''.join(train['Target'])))
chars

[' ',
 '"',
 '#',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 '\\',
 '^',
 '_',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '|',
 '~']

In [4]:
# Duplicate / near-duplicate lines across train - if many, a plain random
# train/val split may leak the same source line into both sides.
train['Target'].value_counts().head(20)

Target
Sealed and delivered in the presence of                                             3
Sealed and delivered in the prsence of                                              3
By this publique Act and Instrument of protest                                      2
Signed Sealed and delivered in the presence of us                                   2
for or by reason of the said Whiston not loading of the said twelve                 2
To all Christian People to whome this present writeing shall come                   2
This done and protested the day and year aforesaid                                  2
To all Christian People to whome these presents shall                               2
By this public act and Instrument of protest be it known                            2
date the Sixth day of July One thousand Six hundred                                 2
hundred thousand pounds of Merchantable                                             1
two store houses and land and ffurther the said

## Follow-up: are near-duplicate typo variants (e.g. "prsence" vs "presence") label noise or genuine period spelling?

The duplicate-count cell above surfaced two variants of the same boilerplate phrase: "Sealed and delivered in the **presence** of" and "Sealed and delivered in the **prsence** of" (missing the first `e`). Checked the source images for both — `zF6Y0Q8CW0z7x0EU` (labeled `prsence`) and `UUMGGElAnTqRKlfO` (labeled `presence`) both clearly show the handwritten word "presence" spelled out in full; there's no visible abbreviation or missing letter in the ink itself.

In [5]:
n_prsence = train['Target'].str.contains('prsence', case=False, na=False).sum()
n_presence = train['Target'].str.contains(r'presence', case=False, na=False, regex=True).sum()
n_prsence, n_presence

(19, 0)

`zF6Y0Q8CW0z7x0EU` (label: `prsence`) vs `UUMGGElAnTqRKlfO` (label: `presence`) — same handwritten word, different transcription:

![prsence example](../data/raw/images/zF6Y0Q8CW0z7x0EU.jpg)
![presence example](../data/raw/images/UUMGGElAnTqRKlfO.jpg)

**Correction (on closer inspection): the handwriting is genuinely ambiguous, not a clear-cut typo.** An earlier pass through this notebook claimed the image `zF6Y0Q8CW0z7x0EU` "unambiguously" shows "presence" written in full — a closer look (and a second opinion) shows that's an overclaim: the cursive ligature between `p` and `s` is compressed and could plausibly be read either way. `notebooks/04_eda_label_noise.ipynb` explored this further (including a per-image review of all 19 `prsence`-labeled rows) and even briefly added a targeted `TARGET_CORRECTIONS` fix in `src/data.py` on the strength of that "unambiguous" claim — that fix was reverted once the underlying visual evidence turned out not to hold up.

**Decision: do not attempt to auto-correct/normalize target spelling anywhere in the pipeline.** 19 rows contain `prsence`, 17 contain `presence` — common enough that it isn't a rare one-off, and even if it were a transcriber inconsistency rather than genuine period variation, we can't verify whether the hidden test set has the same kind of inconsistency. `src/evaluate.py` scores against whatever reference text Zindi provides, so "fixing" predictions to standard spelling could reduce CER/WER against a noisy reference rather than improve it. Treat this as a documented, unresolved ambiguity and a possible small ceiling on achievable CER/WER, not something to engineer around.